# Evaluate a tool-using agent before connecting Claude

Build a bounded tool dispatcher and a trajectory evaluator for a fictional equipment desk, then connect the same read-only contract to an optional Claude Messages API loop.

**Audience:** Python developers building and evaluating enterprise AI applications. Familiarity with dictionaries, functions and JSON is assumed.

**Learning objectives:** enforce a read-only tool-name and argument allowlist; evaluate tool trajectories separately from returned facts; exercise failure cases before optionally running a bounded Claude tool loop.

**Prerequisites:** Understand Python dictionaries, functions, exceptions, and JSON. Local fixtures need only the standard library. The optional Claude branch requires the audited Anthropic service profile, an approved model, and an API credential supplied outside the notebook.

**Setup:** use patched Python 3.12 (3.12.16 or newer) and the audited environment in the [collection README](../../README.md). Run cells in order in a fresh kernel. All default data is newly invented for this tutorial. No customer records, private repositories, credentials, downloads, or paid calls are required. Optional integrations are disabled in their own cells and never enabled by an environment variable alone.

**Execution boundary:** the results below describe the local computation shown in this notebook. They do not establish hosted-model quality or production integration success.

For optional Anthropic or Phoenix integration, install the [audited optional service profile](../../_requirements/optional-services.lock.txt) into the same isolated environment. HTTPX is included in the base environment. An installed package does not imply a live service has been verified.


Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../AI_DISCLOSURE.md), [license](../../LICENSE), and [attribution](../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. Establish a small tool contract

The tools expose only fictional inventory and policy. A caller cannot name an arbitrary Python function or ask the dispatcher to execute code. The schema rejects extra arguments instead of forwarding them to a backend.


In [ ]:
import json

INVENTORY = {
    "camera": {"available": 3, "loan_days": 14},
    "microphone": {"available": 0, "loan_days": 7},
}
TOOLS = [
    {
        "name": "inventory",
        "description": "Look up availability for one equipment type.",
        "input_schema": {
            "type": "object",
            "properties": {"item": {"type": "string", "enum": ["camera", "microphone"]}},
            "required": ["item"],
            "additionalProperties": False,
        },
    },
    {
        "name": "loan_policy",
        "description": "Look up the maximum loan duration for one equipment type.",
        "input_schema": {
            "type": "object",
            "properties": {"item": {"type": "string", "enum": ["camera", "microphone"]}},
            "required": ["item"],
            "additionalProperties": False,
        },
    },
]


def dispatch(name, arguments):
    if not isinstance(name, str) or name not in {"inventory", "loan_policy"}:
        raise ValueError("Tool is not permitted")
    if (
        not isinstance(arguments, dict)
        or set(arguments) != {"item"}
        or not isinstance(arguments["item"], str)
        or arguments["item"] not in INVENTORY
    ):
        raise ValueError("Invalid tool arguments")
    row = INVENTORY[arguments["item"]]
    return (
        {"available": row["available"]} if name == "inventory" else {"loan_days": row["loan_days"]}
    )


assert dispatch("inventory", {"item": "camera"}) == {"available": 3}

## 2. Run realistic local trajectories

The planner fixtures below represent proposed model decisions; they are not generated Claude responses. The dispatcher, tool computations, transcript construction and evaluator are real local code. Keep that distinction when reporting results.


In [ ]:
CASES = [
    {
        "id": "availability",
        "question": "How many cameras are available?",
        "plan": [("inventory", {"item": "camera"})],
        "expected_calls": [("inventory", {"item": "camera"})],
        "expected_output": {"available": 3},
    },
    {
        "id": "duration",
        "question": "How long can I borrow a microphone?",
        "plan": [("loan_policy", {"item": "microphone"})],
        "expected_calls": [("loan_policy", {"item": "microphone"})],
        "expected_output": {"loan_days": 7},
    },
    {
        "id": "combined",
        "question": "Can I borrow a camera, and for how long?",
        "plan": [
            ("inventory", {"item": "camera"}),
            ("loan_policy", {"item": "camera"}),
        ],
        "expected_calls": [
            ("inventory", {"item": "camera"}),
            ("loan_policy", {"item": "camera"}),
        ],
        "expected_output": {"available": 3, "loan_days": 14},
    },
]


def run_plan(plan, max_calls=4):
    if len(plan) > max_calls:
        raise ValueError("Tool-call budget exceeded")
    trace = []
    facts = {}
    for index, (name, arguments) in enumerate(plan):
        result = dispatch(name, arguments)
        facts.update(result)
        trace.append(
            {
                "call_id": f"call-{index}",
                "tool": name,
                "arguments": dict(arguments),
                "result": result,
            }
        )
    return {"trace": trace, "facts": facts}


def score(case, run):
    calls = [(r["tool"], r["arguments"]) for r in run["trace"]]
    return {
        "case_id": case["id"],
        "trajectory_correct": calls == case["expected_calls"],
        "facts_correct": run["facts"] == case["expected_output"],
        "tool_calls": len(calls),
    }


results = [score(c, run_plan(c["plan"])) for c in CASES]
assert all(r["trajectory_correct"] and r["facts_correct"] for r in results)
print(json.dumps(results, indent=2))

## 3. Prove failures are visible

A correct-looking final number is not enough when the wrong tool was used. Mutating one argument must fail the independently stored expectations. Extra arguments, write tools, and overlong plans must fail before side effects.


In [ ]:
mutated = run_plan([("loan_policy", {"item": "camera"})])
assert not score(CASES[1], mutated)["facts_correct"]
assert not score(CASES[1], mutated)["trajectory_correct"]
rejected = 0
for name, args in [
    ("delete_inventory", {"item": "camera"}),
    ("inventory", {"item": "camera", "admin": True}),
    ("inventory", {"item": "server"}),
    ("inventory", {"item": ["camera"]}),
    ("inventory", {"item": {"name": "camera"}}),
    (["inventory"], {"item": "camera"}),
]:
    try:
        dispatch(name, args)
    except ValueError:
        rejected += 1
    else:
        raise AssertionError("Invalid tool call accepted")
try:
    run_plan(CASES[0]["plan"] * 5)
except ValueError:
    rejected += 1
else:
    raise AssertionError("Budget was not enforced")
assert rejected == 7
print({"invalid_calls_or_budgets_rejected": rejected, "mutated_trajectory_detected": True})

## 4. Optional: execute a bounded Claude tool loop

Install the audited anthropic SDK profile, supply ANTHROPIC_API_KEY and an available ANTHROPIC_MODEL through your environment, then explicitly opt in. This loop sends only the invented question and local tool results. It limits rounds and total calls, preserves complete assistant content, and does not execute model-produced Python.

The audited Anthropic SDK uses `httpx2` for its HTTP client. Keep its verified TLS transport and explicit official endpoint; Phoenix in the separate experiment tutorial still uses `httpx`. The client context closes connections on success and failure.


Provider-request and SDK parsing failures expose only an HTTP status or a generic error. Raw provider diagnostics may contain private values, so they are withheld from notebook outputs. Local schema and evidence-validation errors remain explicit.


In [ ]:
RUN_LIVE = False
if RUN_LIVE:
    import os

    import httpx2
    from anthropic import Anthropic

    with Anthropic(
        api_key=os.environ["ANTHROPIC_API_KEY"],
        base_url="https://api.anthropic.com",
        timeout=60,
        max_retries=0,
        http_client=httpx2.Client(timeout=60, verify=True, trust_env=False, follow_redirects=False),
    ) as client:
        messages = [{"role": "user", "content": CASES[2]["question"]}]
        live_trace = []
        for turn in range(5):
            try:
                response = client.messages.create(
                    model=os.environ["ANTHROPIC_MODEL"],
                    max_tokens=600,
                    tools=TOOLS,
                    messages=messages,
                )
            except Exception as exc:
                status = getattr(exc, "status_code", None)
                if type(status) is int:
                    raise RuntimeError(f"Anthropic request failed with HTTP {status}") from None
                raise RuntimeError(
                    "Anthropic request failed; provider diagnostics were withheld"
                ) from None
            messages.append({"role": "assistant", "content": response.content})
            calls = [b for b in response.content if b.type == "tool_use"]
            if not calls:
                if response.stop_reason != "end_turn":
                    raise RuntimeError("Provider did not finish normally")
                break
            results = []
            for call in calls:
                if len(live_trace) >= 4:
                    raise RuntimeError("Live tool-call budget exceeded")
                result = dispatch(call.name, call.input)
                live_trace.append({"tool": call.name, "arguments": call.input, "result": result})
                results.append(
                    {
                        "type": "tool_result",
                        "tool_use_id": call.id,
                        "content": json.dumps(result),
                    }
                )
            messages.append({"role": "user", "content": results})
        else:
            raise RuntimeError("Agent exceeded round budget")
        facts = {k: v for event in live_trace for k, v in event["result"].items()}
        print(score(CASES[2], {"trace": live_trace, "facts": facts}))
else:
    print("Claude was not called. Local fixtures test contracts and evaluators only.")

## Expected local results

All three local trajectory fixtures satisfy both tool and fact expectations. The mutated argument fails its checks; seven invalid tool/budget requests are rejected. Claude remains uncalled unless explicitly enabled.

**Try next:** allow multiple valid tool orders and compare sequence matching with a partial-order evaluator.


## Interpretation, limitations, and cleanup

Exact trajectory matching is deliberately strict; in a real system multiple orders may be valid. Tool facts do not prove that the final natural-language answer expresses those facts faithfully. Add a separate final-answer evaluator and human review, test refusals and unknown equipment, and track latency/cost on repeated live runs. Client-side tool validation is not a substitute for authorization in the real backend.

The default run keeps its artifacts in memory. Restart the kernel to release them. If you opt into a remote integration, review and remove its synthetic test records using that service's documented controls; this notebook does not delete remote data. Never save credential values or private provider output into notebook cells.


## References and provenance

All scenario text, values, evaluation cases and tutorial code were authored for this public example. No external dataset is bundled. Product names identify optional integrations and do not imply endorsement. API references were checked on 2026-10-01.

- [Claude client tool lifecycle](https://platform.claude.com/docs/claude/docs/tool-use)
- [Claude Python Messages API](https://platform.claude.com/docs/en/api/python/messages/create)
